# MVP — Análise de Risco, Custo e Tempo para Implementação de Nova Unidade
## Empresa de Armazenamento de Dados em Nuvem

**Aluno(a):** Ana Luisa Breide Pessôa Guerra
**Matrícula:** 231013289
**Curso/Disciplina:** Sistemas de Suporte à Decisão (SSD)
**Data:** 31/08/2026

---

### Objetivo

Construir um **MVP de suporte à decisão** que avalie **risco, custo e tempo** na
implementação de uma nova unidade (data center / ponto de presença) por uma empresa de
armazenamento de dados em nuvem, a partir de uma base real do Kaggle, e que entregue ao
tomador de decisão **insights acionáveis** e um **roteiro de implementação**.


## Nota sobre a base de dados

O Kaggle não possui uma base pública específica sobre "implementação de unidades de data
center para empresas de nuvem". Este MVP usa a base real **BIM-AI Integrated Dataset**
(Kaggle, usuário `ziya07`) — 1.000 projetos de engenharia civil (edifícios, pontes, túneis,
represas e rodovias) com dados reais de estrutura de decisão: **custo planejado x realizado**,
**prazo planejado x realizado**, **indicadores de segurança/risco** e **variáveis de
monitoramento estrutural** (vibração, trincas, capacidade de carga, sensores de imagem, etc.).

Projetos de construção de uma instalação física compartilham a mesma estrutura de decisão que
a abertura de uma nova unidade de armazenamento em nuvem — orçamento planejado x realizado,
cronograma planejado x realizado e eventos de risco. Por isso a base é usada como **proxy
analítico**, e dentro dela o tipo de projeto **"Building" (edificação)** é tratado como o mais
próximo estruturalmente de um data center, recebendo atenção especial nas seções de
comparação. Essa limitação (proxy, não uma base literal de data centers) é reafirmada na
conclusão.


## 0. Setup e Importações

In [ ]:
!pip install -q scikit-learn scipy seaborn nbformat >/dev/null 2>&1

import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from scipy.stats import shapiro, chi2_contingency, ks_2samp, zscore

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

from datetime import datetime

SEED = 42
np.random.seed(SEED)

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 110

ALUNO_NOME = "Ana Luisa Breide Pessôa Guerra"
ALUNO_MATRICULA = "231013289"
DISCIPLINA = "Sistemas de Suporte à Decisão (SSD)"
DATA_EXECUCAO = datetime.now().strftime("%d/%m/%Y %H:%M")

print(f"Aluno: {ALUNO_NOME} | Matrícula: {ALUNO_MATRICULA}")
print(f"Disciplina: {DISCIPLINA}")
print(f"Execução em: {DATA_EXECUCAO}")


## 1. Carregamento dos Dados

O arquivo `bim_ai_civil_engineering_dataset.csv` (extraído do `archive.zip` do Kaggle) deve
estar na mesma pasta deste notebook no Colab. Se não estiver, a célula abaixo abre
automaticamente o seletor de upload do Colab.


In [ ]:
DATA_PATH = "bim_ai_civil_engineering_dataset.csv"

if not os.path.exists(DATA_PATH):
    try:
        from google.colab import files
        print("Arquivo não encontrado nesta pasta. Faça o upload do CSV do Kaggle:")
        uploaded = files.upload()
        DATA_PATH = list(uploaded.keys())[0]
    except ImportError:
        raise FileNotFoundError(
            f"'{DATA_PATH}' não encontrado. Faça upload do arquivo na pasta do notebook."
        )

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
df.head()


In [ ]:
print("Colunas:", list(df.columns))
print("\nTipos de dado:")
print(df.dtypes)
print("\nValores ausentes por coluna:", df.isna().sum().sum(), "no total")


## 2. Composição da Base (Variáveis Categóricas)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

sns.countplot(data=df, x="Project_Type", ax=axes[0, 0],
              order=df["Project_Type"].value_counts().index, palette="deep")
axes[0, 0].set_title("Projetos por Tipo")
axes[0, 0].tick_params(axis="x", rotation=20)

sns.countplot(data=df, x="Location", ax=axes[0, 1],
              order=df["Location"].value_counts().index, palette="deep")
axes[0, 1].set_title("Projetos por Localização")
axes[0, 1].tick_params(axis="x", rotation=20)

sns.countplot(data=df, x="Weather_Condition", ax=axes[1, 0],
              order=df["Weather_Condition"].value_counts().index, palette="deep")
axes[1, 0].set_title("Projetos por Condição Climática")
axes[1, 0].tick_params(axis="x", rotation=20)

sns.countplot(data=df, x="Risk_Level", ax=axes[1, 1],
              order=["Low", "Medium", "High"],
              palette={"Low": "seagreen", "Medium": "goldenrod", "High": "firebrick"})
axes[1, 1].set_title("Projetos por Nível de Risco")

plt.tight_layout()
plt.show()

print(df["Risk_Level"].value_counts(normalize=True).round(3) * 100)


## 3. Engenharia de Variáveis

A base já traz `Cost_Overrun` (Actual - Planned) e `Schedule_Deviation` (Actual - Planned em
dias). Aqui elas são convertidas para **percentual**, métrica comparável entre projetos de
portes diferentes — essencial para comparar um projeto de data center de qualquer porte.


In [ ]:
df["cost_overrun_pct"] = (df["Cost_Overrun"] / df["Planned_Cost"]) * 100
df["schedule_overrun_pct"] = (df["Schedule_Deviation"] / df["Planned_Duration"]) * 100

metricas = ["cost_overrun_pct", "schedule_overrun_pct", "Safety_Risk_Score",
            "Accident_Count", "Completion_Percentage"]

df[metricas].describe().round(2)


## 4. Estatística Descritiva e Distribuições

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 8))
cols_viz = ["cost_overrun_pct", "schedule_overrun_pct", "Safety_Risk_Score"]

for i, col in enumerate(cols_viz):
    sns.histplot(df[col], kde=True, ax=axes[0, i], color=sns.color_palette()[i])
    axes[0, i].axvline(df[col].mean(), color="black", linestyle="--", linewidth=1)
    axes[0, i].set_title(f"Distribuição — {col}")

    sns.boxplot(y=df[col], ax=axes[1, i], color=sns.color_palette()[i])
    axes[1, i].set_title(f"Boxplot — {col}")

plt.tight_layout()
plt.show()

resumo = df[metricas].agg(["mean", "median", "std", "min", "max", "skew"]).T.round(2)
resumo.columns = ["média", "mediana", "desvio_padrão", "mínimo", "máximo", "assimetria"]
resumo


## 5. Testes de Normalidade (Shapiro-Wilk) e QQ-Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
resultados_shapiro = []

for i, col in enumerate(cols_viz):
    stat, p = shapiro(df[col])
    resultados_shapiro.append({
        "variável": col, "estatística_W": round(stat, 4), "p_valor": round(p, 6),
        "normal_(alpha=0.05)": "Sim" if p > 0.05 else "Não"
    })
    stats.probplot(df[col], dist="norm", plot=axes[i])
    axes[i].set_title(f"QQ-Plot — {col}")

plt.tight_layout()
plt.show()

pd.DataFrame(resultados_shapiro)


## 6. Recorte de Interesse: Projetos do Tipo "Building"

Entre os cinco tipos de projeto da base (Bridge, Building, Dam, Road, Tunnel), **"Building"**
(edificação) é o mais próximo estruturalmente de uma nova unidade de data center — ambos são
construções verticais/prediais, diferente de pontes, túneis, represas e rodovias. Esta seção
compara o subconjunto "Building" com o restante da base para servir de **benchmark direto**
para a decisão da empresa.


In [ ]:
building = df[df["Project_Type"] == "Building"]
resto = df[df["Project_Type"] != "Building"]

print(f"Projetos tipo Building: {len(building)} de {len(df)} ({len(building)/len(df):.1%})")
print()

comparacao = pd.DataFrame({
    "Building (média)": building[metricas].mean(),
    "Demais tipos (média)": resto[metricas].mean(),
}).round(2)
comparacao["diferença"] = (comparacao["Building (média)"] - comparacao["Demais tipos (média)"]).round(2)
print(comparacao)

print("\nDistribuição de risco em projetos Building:")
print((building["Risk_Level"].value_counts(normalize=True) * 100).round(1))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, col in zip(axes, ["cost_overrun_pct", "schedule_overrun_pct"]):
    sns.boxplot(data=df.assign(eh_building=np.where(df["Project_Type"] == "Building", "Building", "Demais tipos")),
                x="eh_building", y=col, ax=ax, palette={"Building": "#4C72B0", "Demais tipos": "#B0B0B0"})
    ax.set_title(f"{col}: Building vs. Demais")
plt.tight_layout()
plt.show()


## 7. Testes Qui-Quadrado: O Que Está (e o Que Não Está) Associado ao Risco

Testa-se se o **nível de risco** (Low/Medium/High) está estatisticamente associado ao
**tipo de projeto**, à **localização** ou às **condições climáticas** — três fatores que a
empresa poderia considerar ao escolher onde/como construir a nova unidade.


In [ ]:
testes_qui = []

for coluna in ["Project_Type", "Location", "Weather_Condition"]:
    tabela = pd.crosstab(df[coluna], df["Risk_Level"])
    chi2, p, dof, esperado = chi2_contingency(tabela)
    testes_qui.append({
        "variável": coluna, "chi2": round(chi2, 3), "p_valor": round(p, 4),
        "graus_liberdade": dof,
        "associação_significativa_(alpha=0.05)": "Sim" if p < 0.05 else "Não"
    })

resultado_qui = pd.DataFrame(testes_qui)
print(resultado_qui)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, coluna in zip(axes, ["Project_Type", "Location", "Weather_Condition"]):
    tabela = pd.crosstab(df[coluna], df["Risk_Level"])[["Low", "Medium", "High"]]
    sns.heatmap(tabela, annot=True, fmt="d", cmap="Blues", ax=ax)
    ax.set_title(f"{coluna} x Risco")
plt.tight_layout()
plt.show()

resultado_qui


## 8. Correlação entre Custo, Tempo, Risco e Variáveis de Monitoramento

In [ ]:
corr_cols = ["Planned_Cost", "Actual_Cost", "cost_overrun_pct",
             "Planned_Duration", "Actual_Duration", "schedule_overrun_pct",
             "Safety_Risk_Score", "Accident_Count", "Vibration_Level", "Crack_Width",
             "Load_Bearing_Capacity", "Completion_Percentage"]
corr = df[corr_cols].corr()

plt.figure(figsize=(11, 9))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Matriz de Correlação")
plt.tight_layout()
plt.show()

corr


## 9. Teste de Kolmogorov-Smirnov

Compara a distribuição de **desvio percentual de custo** dos projetos tipo **Building** contra
a dos **demais tipos**, para checar se o perfil de risco financeiro de uma edificação é
estatisticamente diferente do restante da base — pergunta direta para calibrar a expectativa
da empresa sobre a nova unidade.


In [ ]:
ks_stat, ks_p = ks_2samp(building["cost_overrun_pct"], resto["cost_overrun_pct"])

print("Teste KS — cost_overrun_pct: Building vs. Demais tipos")
print(f"  estatística D = {ks_stat:.4f} | p-valor = {ks_p:.4f}")
print("  Mesma distribuição (alpha=0.05):", "Sim" if ks_p > 0.05 else "Não")

plt.figure(figsize=(8, 5))
sns.ecdfplot(building["cost_overrun_pct"], label="Building")
sns.ecdfplot(resto["cost_overrun_pct"], label="Demais tipos")
plt.legend()
plt.title("ECDF — Desvio de Custo (%): Building vs. Demais")
plt.tight_layout()
plt.show()


## 10. Detecção de Outliers em Projetos "Building" (IQR e Z-Score)

Projetos-edificação com desvios extremos de custo/prazo são os **estudos de caso de maior
risco** para a decisão de abrir a nova unidade — o que pode ter dado errado neles é o que a
empresa mais precisa antecipar.


In [ ]:
def outliers_iqr(serie):
    q1, q3 = serie.quantile([0.25, 0.75])
    iqr = q3 - q1
    lim_inf, lim_sup = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return (serie < lim_inf) | (serie > lim_sup)

building = building.copy()
building["outlier_iqr_custo"] = outliers_iqr(building["cost_overrun_pct"])
building["outlier_iqr_prazo"] = outliers_iqr(building["schedule_overrun_pct"])
building["zscore_custo"] = zscore(building["cost_overrun_pct"])
building["zscore_prazo"] = zscore(building["schedule_overrun_pct"])
building["outlier_zscore"] = (building["zscore_custo"].abs() > 3) | (building["zscore_prazo"].abs() > 3)

print("Outliers (IQR) — desvio de custo:", building["outlier_iqr_custo"].sum())
print("Outliers (IQR) — desvio de prazo:", building["outlier_iqr_prazo"].sum())
print("Outliers (Z-score, |z|>3):", building["outlier_zscore"].sum())

piores_casos = building.sort_values("cost_overrun_pct", ascending=False).head(5)
piores_casos[["Project_ID", "Location", "Planned_Cost", "Actual_Cost", "cost_overrun_pct",
              "schedule_overrun_pct", "Safety_Risk_Score", "Risk_Level"]]


## 11. Segmentação de Perfis de Implementação (K-Means + PCA)

Agrupamento não supervisionado de **toda a base** (não só Building, para ter volume
estatístico suficiente) em perfis de implementação, combinando desvio de custo, desvio de
prazo, risco de segurança, número de acidentes e percentual de conclusão. Cada cluster é um
"arquétipo" de projeto — insumo direto para saber que tipo de cenário a empresa deve esperar
ou evitar.


In [ ]:
features = ["cost_overrun_pct", "schedule_overrun_pct", "Safety_Risk_Score",
            "Accident_Count", "Completion_Percentage"]
X_scaled = StandardScaler().fit_transform(df[features])

scores_silhueta = {}
for k in range(2, 6):
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    labels = km.fit_predict(X_scaled)
    scores_silhueta[k] = silhouette_score(X_scaled, labels)

k_otimo = max(scores_silhueta, key=scores_silhueta.get)
print("Silhouette score por k:", {k: round(v, 3) for k, v in scores_silhueta.items()})
print("k ótimo escolhido:", k_otimo)
print("\n[Nota] Silhouette scores nesta faixa (baixos, ~0.15) indicam clusters com separação")
print("fraca — a base tem baixa correlação entre variáveis (ver Seção 8), então os grupos")
print("abaixo devem ser lidos como tendências direcionais, não fronteiras rígidas.")

kmeans = KMeans(n_clusters=k_otimo, random_state=SEED, n_init=10)
df["cluster"] = kmeans.fit_predict(X_scaled)

pca = PCA(n_components=2, random_state=SEED)
componentes = pca.fit_transform(X_scaled)
df["pca_1"], df["pca_2"] = componentes[:, 0], componentes[:, 1]
print(f"\nVariância explicada pelas 2 componentes: {pca.explained_variance_ratio_.sum():.2%}")

plt.figure(figsize=(8, 6))
sns.scatterplot(data=df, x="pca_1", y="pca_2", hue="cluster", palette="deep", s=50, alpha=0.7)
plt.title(f"Segmentação de Perfis de Implementação — PCA (k={k_otimo})")
plt.xlabel("Componente 1")
plt.ylabel("Componente 2")
plt.tight_layout()
plt.show()


In [ ]:
perfil_clusters = df.groupby("cluster")[features].mean().round(2)
perfil_clusters["n_projetos"] = df["cluster"].value_counts().sort_index()
print(perfil_clusters)

print("\nComposição de Risk_Level por cluster (%):")
print((pd.crosstab(df["cluster"], df["Risk_Level"], normalize="index") * 100).round(1))


## 12. Score de Suporte à Decisão

Score composto (0–100, quanto menor melhor) combinando de forma normalizada o desvio de
custo, o desvio de prazo, o risco de segurança e a acidentalidade de cada cluster — permite
comparar rapidamente os perfis e sinalizar qual a empresa deve buscar (ou mitigar) ao planejar
a nova unidade.


In [ ]:
def normalizar(s):
    if s.max() == s.min():
        return pd.Series(50.0, index=s.index)
    return (s - s.min()) / (s.max() - s.min()) * 100

perfil_score = perfil_clusters.copy()
perfil_score["custo_norm"] = normalizar(perfil_score["cost_overrun_pct"])
perfil_score["prazo_norm"] = normalizar(perfil_score["schedule_overrun_pct"])
perfil_score["seguranca_norm"] = normalizar(perfil_score["Safety_Risk_Score"])
perfil_score["acidentes_norm"] = normalizar(perfil_score["Accident_Count"])

peso_custo, peso_prazo, peso_seguranca, peso_acidentes = 0.25, 0.25, 0.30, 0.20
perfil_score["score_decisao"] = (
    perfil_score["custo_norm"] * peso_custo +
    perfil_score["prazo_norm"] * peso_prazo +
    perfil_score["seguranca_norm"] * peso_seguranca +
    perfil_score["acidentes_norm"] * peso_acidentes
).round(1)

perfil_score_ordenado = perfil_score.sort_values("score_decisao")

print("Ranking de perfis de implementação (menor score = mais favorável à expansão):\n")
print(perfil_score_ordenado[["n_projetos", "cost_overrun_pct", "schedule_overrun_pct",
                              "Safety_Risk_Score", "Accident_Count", "score_decisao"]])

melhor_perfil = perfil_score_ordenado.index[0]
pior_perfil = perfil_score_ordenado.index[-1]
print(f"\nPerfil mais favorável à decisão de expansão: cluster {melhor_perfil}")
print(f"Perfil que exige maior atenção/mitigação de risco: cluster {pior_perfil}")


## 13. Síntese de Insights para o Tomador de Decisão

A célula abaixo gera automaticamente um resumo executivo a partir dos números calculados
acima — não são conclusões fixas: se a base for atualizada, o texto se atualiza junto.


In [ ]:
media_custo = df["cost_overrun_pct"].mean()
mediana_custo = df["cost_overrun_pct"].median()
media_prazo = df["schedule_overrun_pct"].mean()
mediana_prazo = df["schedule_overrun_pct"].median()

pct_alto_risco = (df["Risk_Level"] == "High").mean() * 100

corr_risco_custo = df[["Safety_Risk_Score", "cost_overrun_pct"]].corr().iloc[0, 1]
corr_risco_prazo = df[["Safety_Risk_Score", "schedule_overrun_pct"]].corr().iloc[0, 1]

sig_tipo = resultado_qui.loc[resultado_qui["variável"] == "Project_Type", "associação_significativa_(alpha=0.05)"].values[0]
sig_local = resultado_qui.loc[resultado_qui["variável"] == "Location", "associação_significativa_(alpha=0.05)"].values[0]
sig_clima = resultado_qui.loc[resultado_qui["variável"] == "Weather_Condition", "associação_significativa_(alpha=0.05)"].values[0]

custo_building = building["cost_overrun_pct"].mean()
prazo_building = building["schedule_overrun_pct"].mean()
pct_alto_risco_building = (building["Risk_Level"] == "High").mean() * 100

print("=" * 78)
print("RESUMO EXECUTIVO — INSIGHTS PARA O TOMADOR DE DECISÃO")
print("=" * 78)

print(f'''
1. EXPOSIÇÃO FINANCEIRA E DE PRAZO
   Em média, os projetos da base estouram o orçamento em {media_custo:.1f}% (mediana
   {mediana_custo:.1f}%) e o prazo em {media_prazo:.1f}% (mediana {mediana_prazo:.1f}%).
   Recomendação prática: a empresa deveria reservar uma contingência orçamentária de
   pelo menos {mediana_custo:.0f}% e um buffer de cronograma de pelo menos
   {mediana_prazo:.0f}% ao planejar a nova unidade, em vez de tratar o orçamento/prazo
   inicial como número final.

2. RISCO DE SEGURANÇA É UMA DIMENSÃO SEPARADA DE CUSTO/PRAZO
   O Safety_Risk_Score correlaciona apenas {corr_risco_custo:.2f} com o desvio de custo e
   {corr_risco_prazo:.2f} com o desvio de prazo — ou seja, um projeto pode estourar
   orçamento/prazo sem ter risco de segurança elevado, e vice-versa. Recomendação: a
   empresa precisa de DUAS trilhas de acompanhamento independentes (financeira/cronograma
   e segurança/estrutural), não um único painel de risco.

3. TIPO DE PROJETO, LOCALIZAÇÃO E CLIMA NÃO EXPLICAM O NÍVEL DE RISCO
   Os testes Qui-Quadrado não encontraram associação estatisticamente significativa entre
   Risk_Level e tipo de projeto (significativo: {sig_tipo}), localização (significativo:
   {sig_local}) nem clima (significativo: {sig_clima}). Recomendação: a empresa NÃO deve
   assumir que escolher certa região ou "tipo" de construção automaticamente reduz o risco
   — o risco parece ser determinado por fatores de execução do projeto, não por onde/o que
   é construído. Isso reforça a importância de controles de processo (gestão de obra,
   inspeções) em vez de só escolha de local.

4. BENCHMARK ESPECÍFICO PARA UMA NOVA UNIDADE (perfil "Building")
   Projetos do tipo Building (o mais próximo estruturalmente de um data center) têm desvio
   médio de custo de {custo_building:.1f}% e de prazo de {prazo_building:.1f}%, com
   {pct_alto_risco_building:.1f}% classificados como Alto Risco — {'acima' if pct_alto_risco_building > pct_alto_risco else 'abaixo'}
   da média geral da base ({pct_alto_risco:.1f}%). Este é o número de referência mais direto
   para a expectativa de risco da nova unidade.

5. PERFIS DE IMPLEMENTAÇÃO (Clusters)
   O cluster mais favorável (cluster {melhor_perfil}) tem desvio de custo de
   {perfil_score_ordenado.loc[melhor_perfil, 'cost_overrun_pct']:.1f}% e desvio de prazo de
   {perfil_score_ordenado.loc[melhor_perfil, 'schedule_overrun_pct']:.1f}%, contra
   {perfil_score_ordenado.loc[pior_perfil, 'cost_overrun_pct']:.1f}% e
   {perfil_score_ordenado.loc[pior_perfil, 'schedule_overrun_pct']:.1f}% no cluster de maior
   atenção (cluster {pior_perfil}). Isso mostra que existe uma diferença prática relevante
   entre "o melhor caso" e "o pior caso" de execução — a empresa deve monitorar esses sinais
   (desvio de custo/prazo, acidentes, score de segurança) desde as primeiras semanas do
   projeto para identificar cedo em qual perfil a nova unidade está se encaixando.
''')

print("=" * 78)


## 14. Roteiro de Implementação — "Como a Empresa Vai se Programar"

Com base nos achados acima, um roteiro prático de como a empresa deveria se programar para
implementar a nova unidade:

1. **Planejamento orçamentário e de cronograma**
   Definir o orçamento e o prazo-base da nova unidade e já embutir a contingência sugerida
   na Seção 13 (insight 1), em vez de tratá-la como reserva "extra" opcional.

2. **Dois comitês/trilhas de acompanhamento**
   Como custo/prazo e segurança são praticamente independentes (insight 2), a governança do
   projeto deve ter indicadores e reuniões de acompanhamento separados para (a) orçamento e
   cronograma e (b) segurança/qualidade estrutural — um problema em uma trilha não é sinal
   automático de problema na outra, mas ambos precisam de atenção simultânea.

3. **Escolha de local com base em critérios operacionais, não de "risco"**
   Como local/clima/tipo de construção não explicam estatisticamente o nível de risco
   (insight 3), a decisão de onde construir deve priorizar critérios como custo de energia,
   conectividade e logística — não a expectativa (não comprovada nesta base) de que uma
   região é "mais segura" para construir.

4. **Usar o perfil "Building" como expectativa inicial de referência**
   Os números da Seção 13 (insight 4) servem como ponto de partida realista para o
   orçamento/cronograma/risco esperado da nova unidade, ajustados pela experiência e
   contexto específico da empresa.

5. **Monitoramento por sinais precoces, não só por marco final**
   Acompanhar desde o início os indicadores usados na segmentação (desvio de custo, desvio
   de prazo, score de segurança, contagem de acidentes) permite identificar rapidamente se o
   projeto está seguindo o perfil "favorável" ou o de "maior atenção" (insight 5), e agir
   antes do estouro se consolidar.


## 15. Exportação dos Resultados (CSV / TXT)

In [ ]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

nome_csv = f"resumo_risco_custo_tempo_{timestamp}.csv"
nome_txt = f"relatorio_mvp_nova_unidade_{timestamp}.txt"

df.to_csv(nome_csv, index=False, encoding="utf-8")

with open(nome_txt, "w", encoding="utf-8") as f:
    f.write("=" * 70 + "\n")
    f.write("MVP - ANALISE DE RISCO, CUSTO E TEMPO PARA NOVA UNIDADE\n")
    f.write("Empresa de Armazenamento de Dados em Nuvem\n")
    f.write("=" * 70 + "\n")
    f.write(f"Aluno: {ALUNO_NOME}\n")
    f.write(f"Matricula: {ALUNO_MATRICULA}\n")
    f.write(f"Disciplina: {DISCIPLINA}\n")
    f.write(f"Data de execucao: {DATA_EXECUCAO}\n")
    f.write(f"Fonte dos dados: BIM-AI Integrated Dataset (Kaggle, ziya07)\n")
    f.write("-" * 70 + "\n\n")

    f.write("RESUMO DESCRITIVO (desvio de custo, desvio de prazo, risco)\n")
    f.write(resumo.to_string() + "\n\n")

    f.write("TESTES QUI-QUADRADO (Risk_Level vs. Tipo/Local/Clima)\n")
    f.write(resultado_qui.to_string() + "\n\n")

    f.write("TESTE KOLMOGOROV-SMIRNOV (Building vs. demais tipos - custo)\n")
    f.write(f"  D = {ks_stat:.4f} | p-valor = {ks_p:.4f}\n\n")

    f.write(f"RANKING DE PERFIS DE IMPLEMENTACAO (k={k_otimo} clusters)\n")
    f.write(perfil_score_ordenado.to_string() + "\n\n")

    f.write(f"Perfil mais favoravel: cluster {melhor_perfil}\n")
    f.write(f"Perfil de maior atencao: cluster {pior_perfil}\n\n")

    f.write("-" * 70 + "\n")
    f.write(f"Relatorio gerado por {ALUNO_NOME} - {DISCIPLINA}\n")
    f.write("=" * 70 + "\n")

print("Arquivos exportados:")
print(" -", nome_csv)
print(" -", nome_txt)


## Conclusão e Limitações

- O pipeline completo (estatística descritiva, testes de normalidade, Qui-quadrado,
  correlação, KS, detecção de outliers e segmentação K-Means/PCA) foi aplicado a uma base
  **real** do Kaggle (BIM-AI Integrated Dataset), reenquadrada como proxy para a decisão de
  abertura de uma nova unidade de armazenamento em nuvem.
- Os principais achados — magnitude de estouro de custo/prazo, independência entre risco de
  segurança e risco financeiro/cronograma, ausência de associação entre risco e
  local/tipo/clima, e a segmentação por perfis de implementação — foram traduzidos em um
  **roteiro prático de implementação** (Seção 14) para o tomador de decisão.
- **Limitação central:** a base é sobre projetos de engenharia civil em geral (edifícios,
  pontes, túneis, represas, rodovias), não especificamente sobre data centers. Os números
  absolutos não devem ser lidos como estimativas de mercado da empresa — o valor está na
  **metodologia de análise**, replicável em qualquer base real de projetos de expansão que a
  empresa venha a coletar sobre suas próprias unidades.
- Os scores de silhueta da segmentação (Seção 11) foram baixos (~0.15), indicando clusters
  com separação estatística fraca — coerente com a baixa correlação encontrada na Seção 8.
  Os perfis identificados devem ser lidos como tendências direcionais, não fronteiras
  rígidas entre "tipos" de projeto.
- `SEED=42` foi usado em todas as etapas aleatórias para garantir reprodutibilidade.

---
**Aluno(a):** Ana Luisa Breide Pessôa Guerra | **Matrícula:** 231013289
**Disciplina:** Sistemas de Suporte à Decisão (SSD) | **Data:** 31/08/2026
